# Brian2

[Brian2](https://brian2.readthedocs.io) is a free, open source simulator for spiking neural networks.
It is widely used in computational neuroscience and supports flexible, equation-oriented neuron models.

```{include} ../../../tmp/examples/brian2/supported_primitives.md
```

## Import a NIR graph to Brian2

The following example converts a NIR graph into a Brian2 simulation.
NIR uses dimensionless (normalized) quantities, so Brian2 is run in `unit_checks=False`
mode to accept plain floating-point parameters directly from NIR.

NIR LIF dynamics:
$$\tau \frac{dv}{dt} = v_{\text{leak}} - v + r \cdot I$$
with spike emitted when $v > v_{\text{threshold}}$ and reset to $v_{\text{reset}}$.

In [ ]:
import numpy as np
from brian2 import NeuronGroup, Synapses, SpikeMonitor, StateMonitor, run, start_scope
import brian2

import nir


def nir_to_brian2(nir_graph, dt=0.0001):
    """Convert a NIR graph to a Brian2 network.

    Parameters
    ----------
    nir_graph : nir.NIRGraph
        The NIR graph to convert.
    dt : float
        Simulation timestep in seconds (default: 0.1 ms).

    Returns
    -------
    dict
        A dictionary with Brian2 objects keyed by NIR node name.
    """
    start_scope()
    brian2.defaultclock.dt = dt * brian2.second

    # Disable unit checking so NIR's dimensionless parameters are accepted
    brian2.prefs.codegen.target = 'numpy'
    brian2.prefs.core.network.default_dt = dt * brian2.second

    groups = {}  # name -> Brian2 object

    # First pass: create NeuronGroups for each computational node
    for name, node in nir_graph.nodes.items():
        if isinstance(node, nir.Input):
            n_neurons = int(np.prod(node.input_type))
            # Placeholder group that can receive external spike injection
            g = NeuronGroup(
                n_neurons,
                'v : 1',
                threshold='v > 0.5',
                reset='v = 0',
                method='exact',
                namespace={},
            )
            groups[name] = g

        elif isinstance(node, nir.LIF):
            n_neurons = int(np.prod(node.tau.shape))
            tau_val = node.tau.flatten()
            r_val = node.r.flatten()
            v_leak_val = node.v_leak.flatten()
            v_thr_val = node.v_threshold.flatten()
            v_reset_val = np.zeros(n_neurons)
            if hasattr(node, 'v_reset') and node.v_reset is not None:
                v_reset_val = node.v_reset.flatten()

            # NIR LIF: tau * dv/dt = (v_leak - v + r * I)
            eqs = '''
            dv/dt = (v_leak - v + r * I) / tau : 1
            I : 1
            v_leak : 1
            r : 1
            tau : 1
            '''
            g = NeuronGroup(
                n_neurons,
                eqs,
                threshold='v > v_threshold',
                reset='v = v_reset',
                method='exact',
                namespace={
                    'v_threshold': v_thr_val[0] if len(set(v_thr_val)) == 1 else v_thr_val,
                    'v_reset': v_reset_val[0] if len(set(v_reset_val)) == 1 else v_reset_val,
                },
            )
            g.tau = tau_val
            g.r = r_val
            g.v_leak = v_leak_val
            g.v = v_leak_val
            groups[name] = g

        elif isinstance(node, nir.IF):
            n_neurons = int(np.prod(node.r.shape))
            r_val = node.r.flatten()
            v_thr_val = node.v_threshold.flatten()
            v_reset_val = np.zeros(n_neurons)
            if hasattr(node, 'v_reset') and node.v_reset is not None:
                v_reset_val = node.v_reset.flatten()

            # NIR IF: r * dv/dt = I (no leak)
            eqs = '''
            dv/dt = I / r : 1
            I : 1
            r : 1
            '''
            g = NeuronGroup(
                n_neurons,
                eqs,
                threshold='v > v_threshold',
                reset='v = v_reset',
                method='euler',
                namespace={
                    'v_threshold': v_thr_val[0] if len(set(v_thr_val)) == 1 else v_thr_val,
                    'v_reset': v_reset_val[0] if len(set(v_reset_val)) == 1 else v_reset_val,
                },
            )
            g.r = r_val
            g.v = 0
            groups[name] = g

        elif isinstance(node, (nir.Affine, nir.Linear)):
            # Affine/Linear nodes are realized as Brian2 Synapses (added in second pass)
            groups[name] = node  # store NIR node; synapses created in second pass

        elif isinstance(node, nir.Output):
            groups[name] = None  # output is handled via monitors

    # Second pass: create Synapses for Affine/Linear nodes
    synapses = {}
    for src_name, dst_name in nir_graph.edges:
        src_node = nir_graph.nodes[src_name]
        dst_node = nir_graph.nodes[dst_name]

        if isinstance(src_node, (nir.Affine, nir.Linear)):
            # Find the pre-synaptic group (the node feeding into this Affine)
            pre_names = [s for s, d in nir_graph.edges if d == src_name]
            if not pre_names:
                continue
            pre_group = groups.get(pre_names[0])
            post_group = groups.get(dst_name)
            if pre_group is None or post_group is None or not isinstance(post_group, NeuronGroup):
                continue

            weight_matrix = src_node.weight

            syn = Synapses(
                pre_group,
                post_group,
                'w : 1',
                on_pre='I_post += w',
            )
            syn.connect()
            # Set weight matrix (rows = post, cols = pre)
            n_pre = weight_matrix.shape[1]
            n_post = weight_matrix.shape[0]
            for j in range(n_post):
                for i in range(n_pre):
                    syn.w[i, j] = weight_matrix[j, i]

            bias = src_node.bias if isinstance(src_node, nir.Affine) else np.zeros(weight_matrix.shape[0])
            # Apply bias by setting initial current
            if hasattr(post_group, 'I'):
                post_group.I = bias

            synapses[f'{src_name}_{dst_name}'] = syn

    return groups, synapses


# --- Example usage ---

# Build a simple NIR graph: Input -> Affine -> LIF -> Output
weight = np.array([[1.5, 0.0], [0.0, 1.5]])
bias = np.array([0.0, 0.0])
tau = np.array([0.01, 0.01])   # 10 ms time constant
r = np.array([1.0, 1.0])
v_leak = np.array([0.0, 0.0])
v_threshold = np.array([1.0, 1.0])

nir_graph = nir.NIRGraph(
    nodes={
        'input': nir.Input(input_type=np.array([2])),
        'affine': nir.Affine(weight=weight, bias=bias),
        'lif': nir.LIF(tau=tau, r=r, v_leak=v_leak, v_threshold=v_threshold),
        'output': nir.Output(output_type=np.array([2])),
    },
    edges=[
        ('input', 'affine'),
        ('affine', 'lif'),
        ('lif', 'output'),
    ],
)

groups, synapses = nir_to_brian2(nir_graph, dt=0.0001)
print('Brian2 groups created:', list(k for k, v in groups.items() if v is not None))
print('Brian2 synapses created:', list(synapses.keys()))


## Export a NIR graph from Brian2

The following example shows how to convert a Brian2 `NeuronGroup` with LIF dynamics
back into a NIR graph.  Because Brian2 uses string equations, the exporter requires
the user to annotate the groups with NIR-compatible parameter names (`tau`, `r`,
`v_leak`, `v_threshold`).

In [ ]:
import numpy as np
from brian2 import NeuronGroup, Synapses, start_scope
import brian2

import nir


def brian2_lif_to_nir(neuron_group, v_threshold, v_reset=None):
    """Export a Brian2 LIF NeuronGroup to a NIR LIF node.

    The NeuronGroup must have parameters ``tau``, ``r``, and ``v_leak``
    (as per-neuron variables or shared scalars).

    Parameters
    ----------
    neuron_group : brian2.NeuronGroup
        Source LIF group.
    v_threshold : float or array_like
        Spike threshold (dimensionless, matching NIR convention).
    v_reset : float or array_like, optional
        Reset potential (default: 0).

    Returns
    -------
    nir.LIF
    """
    N = len(neuron_group)
    tau = np.asarray(neuron_group.tau).flatten()
    r = np.asarray(neuron_group.r).flatten()
    v_leak = np.asarray(neuron_group.v_leak).flatten()
    v_thr = np.broadcast_to(np.asarray(v_threshold), (N,)).copy()
    v_rst = np.zeros(N) if v_reset is None else np.broadcast_to(np.asarray(v_reset), (N,)).copy()
    return nir.LIF(tau=tau, r=r, v_leak=v_leak, v_threshold=v_thr, v_reset=v_rst)


def brian2_synapses_to_nir_affine(synapses, n_pre, n_post):
    """Export a Brian2 all-to-all Synapses object to a NIR Affine node.

    Parameters
    ----------
    synapses : brian2.Synapses
        Fully-connected synapse object with a weight variable ``w``.
    n_pre : int
        Number of pre-synaptic neurons.
    n_post : int
        Number of post-synaptic neurons.

    Returns
    -------
    nir.Affine
    """
    weight = np.zeros((n_post, n_pre))
    for i in range(n_pre):
        for j in range(n_post):
            weight[j, i] = float(synapses.w[i, j])
    bias = np.zeros(n_post)
    return nir.Affine(weight=weight, bias=bias)


# --- Example: build and export a Brian2 LIF population ---

start_scope()
brian2.prefs.codegen.target = 'numpy'

N = 3
eqs = '''
dv/dt = (v_leak - v + r * I) / tau : 1
I : 1
v_leak : 1
r : 1
tau : 1
'''
lif_group = NeuronGroup(
    N, eqs,
    threshold='v > 1.0',
    reset='v = 0.0',
    method='exact',
)
lif_group.tau = [0.02, 0.02, 0.02]  # 20 ms
lif_group.r = [1.0, 1.0, 1.0]
lif_group.v_leak = [0.0, 0.0, 0.0]

# Export to NIR
nir_lif = brian2_lif_to_nir(lif_group, v_threshold=1.0)

nir_graph = nir.NIRGraph(
    nodes={
        'input': nir.Input(input_type=np.array([N])),
        'lif': nir_lif,
        'output': nir.Output(output_type=np.array([N])),
    },
    edges=[('input', 'lif'), ('lif', 'output')],
)

print('Exported NIR graph:')
print(nir_graph)

# Save to file
nir.write('brian2_lif.nir', nir_graph)
print('Saved to brian2_lif.nir')
